### Custom models

The mental model is very similar to creating a neural network in pytorch. Except that there is zeroing the gradients or backprop. The optimizer step is explored.

Only the LLM facing components will be optimized i.e. the __init__ where you would declare
all the layers of your NN. forward is just like the wiring(computation graph portion) 

In [1]:
import dspy
import os
from dotenv import load_dotenv

load_dotenv()

ANTHROPIC_KEY = os.getenv("ANTHROPIC_KEY")
OPENAI_KEY = os.getenv("OPENAI_KEY")

In [2]:
lm = dspy.LM("anthropic/claude-opus-4-6", api_key=ANTHROPIC_KEY)
dspy.configure(lm=lm, cache=False)

In [3]:
#Simple QA Module

class SimpleQAModule(dspy.Module):
    def __init__(self):
        self.answer = dspy.Predict("question -> answer")

    def forward(self, question):
        return self.answer(question=question)

qa_module = SimpleQAModule()
qa_module(question="Why is sky blue?")

Prediction(
    answer="The sky appears blue due to a phenomenon called **Rayleigh scattering**. Here's how it works:\n\n1. **Sunlight composition**: Sunlight (white light) is made up of all colors of the visible spectrum, each with different wavelengths. Blue and violet light have shorter wavelengths, while red and orange light have longer wavelengths.\n\n2. **Interaction with the atmosphere**: When sunlight enters Earth's atmosphere, it collides with gas molecules (mainly nitrogen and oxygen). These molecules are much smaller than the wavelengths of visible light.\n\n3. **Scattering**: Shorter wavelengths of light (blue and violet) are scattered much more efficiently by these tiny molecules than longer wavelengths (red, orange, yellow). Specifically, the amount of scattering is inversely proportional to the fourth power of the wavelength — meaning blue light (~470 nm) is scattered about 5.5 times more than red light (~650 nm).\n\n4. **Why blue and not violet?**: Even though violet li

In [4]:
## Chain 2 LLM calls

class JudgeQA(dspy.Module):
    def __init__(self):
        self.answer = dspy.Predict("question -> answer")
        self.judge = dspy.ChainOfThought("question, answer -> is_correct:bool, reasoning:str")

    def forward(self, question): #This is akin to foward(X)
        result = self.answer(question=question)
        judge_results = self.judge(question=question, answer=result)
        if judge_results.is_correct:
            return f"{result}\n\n{judge_results}"
        return f"Answer is wrong and reasoning is as follows {judge_results.reasoning}"

In [5]:
judge_qa = JudgeQA()
print(judge_qa(question="What is hard truth about life?")) #Subjective question. Dont worry. I am fine :)


Prediction(
    answer="There are many hard truths about life, but here are some of the most significant ones:\n\n1. **Life is not fair.** Bad things happen to good people, and good things happen to bad people. Merit alone doesn't guarantee success or happiness.\n\n2. **Everyone you love will either leave you or die.** Loss is an unavoidable part of the human experience. Every relationship ends — through distance, change, or death.\n\n3. **Time is limited and non-renewable.** You cannot get back a single moment. Every day that passes is gone forever, and none of us knows how many days we have left.\n\n4. **No one is coming to save you.** Ultimately, you are responsible for your own life. While others can help, the burden of building a meaningful life falls on your own shoulders.\n\n5. **Suffering is inevitable.** Pain — physical, emotional, existential — is woven into the fabric of being alive. You cannot avoid it; you can only learn how to navigate it.\n\n6. **Most people are too focu

#### Two-step email drafter
given a customer complaint, first analyze the complaint to identify the core issue and customer emotion, then draft a professional response.

Do both evaluation and optimization

In [6]:
complaints = [
    {"complaint": "I've been waiting 3 weeks for my refund and nobody responds to my emails. This is unacceptable.", "core_issue": "delayed_refund", "emotion": "frustrated"},
    {"complaint": "Your app deleted all my saved data after the last update. I lost 2 years of work.", "core_issue": "data_loss", "emotion": "angry"},
    {"complaint": "I was charged $200 for a feature I never signed up for. Please explain.", "core_issue": "unauthorized_charge", "emotion": "concerned"},
    {"complaint": "The delivery driver left my package in the rain and everything inside is ruined.", "core_issue": "damaged_delivery", "emotion": "upset"},
    {"complaint": "I cancelled my subscription last month but I'm still being billed. This is the third time I've reported this.", "core_issue": "billing_after_cancellation", "emotion": "frustrated"},
    {"complaint": "Your customer support chat told me my issue would be fixed in 24 hours. That was 10 days ago.", "core_issue": "broken_support_promise", "emotion": "disappointed"},
    {"complaint": "The product I received looks nothing like what was advertised on your website.", "core_issue": "misleading_product", "emotion": "disappointed"},
    {"complaint": "I've been a loyal customer for 8 years and you just removed the one feature I use daily without any notice.", "core_issue": "feature_removal", "emotion": "betrayed"},
]

In [7]:
from typing import Literal

full_dataset = [dspy.Example(complaint) for complaint in complaints]

class ComplaintAssesment(dspy.Signature):
    """ Assess a brief description of the complaint and categorize into 2 buckets
        1) core_issue - Helps understand the issue at hand
        2) Emotion - Helps understand the dissatisfaction customer had
    """
    complaint:str = dspy.InputField(desc="Summary of a complaint email")
    core_issue:Literal["delayed_refund", "data_loss", "unauthorized_charge", "damaged_delivery",\
        "billing_after_cancellation","broken_support_promise","misleading_product", "feature_removal"] = dspy.OutputField(desc="Issue described by customer. Choose the most appropriate categroy")
    emotion:Literal["frustrated", "angry", "concerned",\
        "upset", "disappointed", "betrayed"] = dspy.OutputField(desc="Emotion deduced based on complaint")

class EmailDraft(dspy.Signature):
    """Given a brief description of the customer complaint email, core issue and 
        their emotion, compose a draft reponse email that would help resolve the issue
    """

    complaint:str = dspy.InputField(desc="Summary of a complaint email")
    core_issue:str = dspy.InputField(desc="One word describing the core issue at hand")
    emotion:str = dspy.InputField(desc="Emotion potentially the customer had when writing this email")
    response_draft_email:str = dspy.OutputField(desc="A response email that would help resolve customer's complaint")


In [8]:
##The actual module
class EmailAnalyseAndDrafter(dspy.Module):

    def __init__(self):
        self.complaint_assesor = dspy.Predict(signature=ComplaintAssesment)
        self.email_draft = dspy.ChainOfThought(signature=EmailDraft)

    def forward(self, complaint):
        email_analysis_results = self.complaint_assesor(complaint=complaint)
        return self.email_draft(complaint=complaint, core_issue=email_analysis_results.core_issue,
                            emotion=email_analysis_results.emotion
                               ), email_analysis_results
        

In [9]:
two_step_email = EmailAnalyseAndDrafter()
two_step_email(complaint=complaints[0]["complaint"])

(Prediction(
     reasoning='The customer is frustrated because they have been waiting 3 weeks for a refund and have not received any responses to their previous emails. The response needs to: 1) Acknowledge the frustration and apologize sincerely, 2) Take responsibility for the lack of communication, 3) Provide a concrete update or action plan regarding the refund, 4) Give a specific timeline, and 5) Offer a direct point of contact so the customer feels heard and supported.',
     response_draft_email='Subject: Re: Your Refund Request – Immediate Attention\n\nDear Valued Customer,\n\nThank you for reaching out, and I sincerely apologize for the unacceptable delay in both your refund and our response to your previous emails. I completely understand your frustration, and I want you to know that this is not the level of service we strive to provide.\n\nI have personally escalated your refund request to our finance team and can confirm that your refund is now being prioritized. You should